# Lab 1 — Can We Trust the Data?
## Operationalizing a Recruiting Data Product

**Mission:** Lab 2 will rank schools and recommend engagement actions. Before a model can use this data, you must turn it into a trustworthy, reusable data product.

1. `clean_recruiting_events.csv` — one validated row per engagement.
2. `school_summary.csv` — one aggregated row per school.
3. An auditable exception record explaining what was excluded and why.

The raw file contains 36 schools, hundreds of repeated events, aliases, misspellings, duplicate IDs, mixed dates, missing keys, and invalid funnel values.

**Learning goals**

- Profile data before transforming it.
- Resolve school and action identities using explicit reference rules.
- Preserve legitimate repeated events while removing duplicates.
- Validate `contacts, appointments, qualified contracts`.
- Produce event-level and school-level artifacts with auditable lineage.
- Direct, inspect, and verify an AI coding assistant in small steps.

> **Use your coding assistant as a teammate.** You are the Navigator (define the goal), Reviewer (inspect the proposal), and Decision-Maker (accept only verified changes). Give it the current cell, the self-check output, and the goal.

Suggested prompt:

> I am working in a classroom Jupyter notebook. First explain what this self-check is testing and any assumptions in the proposed solution. Then suggest the smallest edit to the marked variables. Do not change the data or the test.

> **Operating rule:** A green self-check is evidence, not a substitute for judgment. Record why a transformation is valid and what it cannot prove.

In [ ]:
#@title
from IPython.display import display, Markdown, HTML
display(HTML('<div style="background-color: rgba(128, 128, 128, 0.12); border: 1px solid rgba(128, 128, 128, 0.28); border-radius: 6px; padding: 12px 16px; margin: 8px 0 12px;"><h2 style="margin: 0 0 8px;">0. Setup</h2><p style="margin: 0 0 14px;">Run these setup blocks before profiling the recruiting-event export.</p><h3 style="margin: 0 0 6px;">0.1 Define notebook helpers</h3><p style="margin: 0;">This block defines reusable self-check and mission-checkpoint helpers used throughout the lab.</p></div>'))


0. Setup Run these setup blocks before profiling the recruiting-event export. 0.1 Define notebook helpers This block defines reusable self-check and mission-checkpoint helpers used throughout the lab.

In [ ]:

def check(name, condition, hint=""):
    try:
        passed = bool(condition)
    except Exception as exc:
        passed = False
        hint = f"{hint} ({type(exc).__name__}: {exc})"
    icon = "✅" if passed else "❌"
    print(f"{icon} {name}")
    if not passed and hint:
        print(f"   Hint: {hint}")
    return passed

def mission_header(text):
    display(Markdown(f"> **Mission checkpoint:** {text}"))

0.2 Import packages and load the raw export This block imports the data tools, configures the table display, and loads the fictional raw recruiting-event dataset.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(128, 128, 128, 0.12); border: 1px solid rgba(128, 128, 128, 0.28); border-radius: 6px; padding: 12px 16px; margin: 8px 0 12px;"><h3 style="margin: 0 0 6px;">0.2 Import packages and load the raw export</h3><p style="margin: 0;">This block imports the data tools, configures the table display, and loads the fictional raw recruiting-event dataset.</p></div>'))


🛠️ TODO: Complete the three approved school-label corrections. 🤔 Think: Which label changes are mechanical formatting, and which need an explicit identity decision? 💡 Hint: Use only the three normalized labels shown in the comments. Do not add fuzzy matching.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

raw = pd.read_csv("https://usard-demo.netlify.app/data/raw_recruiting_events.csv")
print(f"Loaded {len(raw):,} raw rows from https://usard-demo.netlify.app/data/raw_recruiting_events.csv")
raw.head()

# Lab A — Make the pipeline observable

A recommender cannot repair an inconsistent pipeline. Start with evidence, then make a small number of explicit, testable decisions.

## A1. Profile before fixing

Before running code, name three acceptance criteria for a model-ready event record. Then pause and predict: how many exact duplicates, missing fields, and suspicious labels do you expect? Which of these would change a downstream recommendation?

In [ ]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(dropna=True),
})
display(profile)
print("Rows:", len(raw))
print("Exact duplicate rows:", raw.duplicated().sum())
print("Distinct raw school labels:", raw["school_name"].nunique(dropna=True))
print("Distinct raw action labels:", raw["action"].nunique(dropna=True))

⬇️ 🔒 Checking cell below: Only run this after completing the school-identity mapping. ⬇️

In [ ]:
check("The raw export contains 494 rows", len(raw) == 494)
check("Twelve exact duplicate rows are visible", raw.duplicated().sum() == 12)
check("Raw labels exceed the 36 real schools", raw["school_name"].nunique() > 36)

🛠️ TODO: Map each alternate action label to one of the six canonical actions. 🤔 Think: Why would leaving one alias unresolved distort the recommender matrix in Lab 2? 💡 Hint: Read the unresolved labels and map them to the canonical action names—not to new names.

## A2. Establish one school identity

Most variants can be normalized mechanically. Three genuine misspellings require explicit decisions against the approved reference list. Complete `MANUAL_SCHOOL_FIXES`; do not use unrestricted fuzzy matching. A plausible match is not an authoritative identity.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;"><p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Complete the three approved school-label corrections.</p><p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> Which label changes are mechanical formatting, and which need an explicit identity decision?</p><div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;"><strong>💡 Hint:</strong> Use only the three normalized labels shown in the comments. Do not add fuzzy matching.</div></div>'))


⬇️ 🔒 Checking cell below: Only run this after completing the action catalog. ⬇️

In [ ]:
school_names = [
    "Lincoln High", "Jefferson High", "Washington High", "Roosevelt High",
    "North County Tech", "Lakeside Academy", "Madison High", "Franklin High",
    "Central High", "Riverside High", "Eastview High", "Westfield High",
    "Pine Ridge High", "Oak Valley High", "Summit High", "Cedar Grove High",
    "Parkview High", "Liberty High", "Monroe High", "Adams High",
    "Hamilton High", "Kennedy High", "Jackson High", "Grant High",
    "Wilson High", "Heritage High", "Valley Tech", "Mountain View High",
    "Harbor High", "Brookside High", "Greenfield High", "Redstone High",
    "Horizon High", "Pioneer High", "Union High", "Victory High",
]
SCHOOL_REFERENCE = {
    name.upper(): (f"S{i:03d}", name)
    for i, name in enumerate(school_names, start=1)
}

MANUAL_SCHOOL_FIXES = {
    # TODO: map these normalized labels:
    # "JEFFRSON HIGH": "JEFFERSON HIGH",
    # "N COUNTY TECHNICAL": "NORTH COUNTY TECH",
    # "LAKESIDE ACAD": "LAKESIDE ACADEMY",
}

def normalize_school_label(value):
    if pd.isna(value) or not str(value).strip():
        return None
    label = re.sub(r"\s+", " ", str(value).strip().replace(".", "")).upper()
    label = MANUAL_SCHOOL_FIXES.get(label, label)
    label = re.sub(r" HIGH SCHOOL$", " HIGH", label)
    label = re.sub(r" HS$", " HIGH", label)
    return label

working = raw.copy()
working["school_label"] = working["school_name"].map(normalize_school_label)
working["school_id"] = working["school_label"].map(lambda x: SCHOOL_REFERENCE.get(x, (None, None))[0])
working["school_name_clean"] = working["school_label"].map(lambda x: SCHOOL_REFERENCE.get(x, (None, None))[1])

unresolved_schools = working.loc[
    working["school_name"].notna() & working["school_id"].isna(), "school_name"
].value_counts()
unresolved_schools

In [ ]:
#@title
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;"><p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only run this after completing the school-identity mapping. <strong>⬇️</strong></p></div>'))


🛠️ TODO: Turn on the duplicate-ID removal policy. 🤔 Think: Why is an identical-looking row different from a duplicated engagement ID? 💡 Hint: Preserve legitimate repeated events; remove only later records with an engagement ID already seen.

In [ ]:
check("All nonblank school labels resolve", unresolved_schools.empty,
      "Complete the three explicit mappings in MANUAL_SCHOOL_FIXES.")
check("Exactly 36 canonical schools are represented", working["school_id"].nunique() == 36)

### Coding-assistant challenge

Ask your coding assistant:

> These are the unresolved school labels, and this is the approved school reference list. Explain which changes are mechanical normalization and which require an explicit business decision. Then suggest only the entries for `MANUAL_SCHOOL_FIXES`. Do not use fuzzy matching, create a new school, or change the reference list.

Read its proposal before applying it. Why would an unrestricted fuzzy match be unsafe in a production recruiting pipeline?

## A3. Establish one action catalog

Complete the alias map. Different spellings of the same action must not become different matrix columns later. In Lab 2, an action label becomes an item in the recommender matrix.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;"><p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Map each alternate action label to one of the six canonical actions.</p><p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> Why would leaving one alias unresolved distort the recommender matrix in Lab 2?</p><div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;"><strong>💡 Hint:</strong> Read the unresolved labels and map them to the canonical action names—not to new names.</div></div>'))


⬇️ 🔒 Checking cell below: Verify the duplicate policy and date parsing before continuing. ⬇️

In [ ]:
canonical_actions = [
    "Cyber Careers Event", "STEM Careers Presentation", "Mechanical Careers Demo",
    "Healthcare Careers Session", "Education Benefits Session", "General Recruiting Table",
]
ACTION_NAME_MAP = {action.upper(): action for action in canonical_actions}
ACTION_NAME_MAP.update({
    # TODO: add aliases such as "STEM PRESENTATION": "STEM Careers Presentation"
})

working["action_label"] = working["action"].map(
    lambda value: None if pd.isna(value) else str(value).strip().upper()
)
working["action_clean"] = working["action_label"].map(ACTION_NAME_MAP)

unresolved_actions = working.loc[
    working["action"].notna() & working["action_clean"].isna(), "action"
].value_counts()
unresolved_actions

In [ ]:
#@title
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;"><p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Only run this after completing the action catalog. <strong>⬇️</strong></p></div>'))


🛠️ TODO: Exclude invalid records from the model-ready artifact. 🤔 Think: Why should invalid rows stay in the exception queue even when they are excluded from the clean dataset? 💡 Hint: Change the policy string to exclude ; the rejection queue is created separately below.

In [ ]:
check("All nonblank action aliases resolve", unresolved_actions.empty,
      "Map singular, abbreviated, and alternate action names to the six canonical actions.")
check("Exactly six canonical actions remain", working["action_clean"].nunique() == 6)

# Lab B — Validate and triage events

## B1. Preserve real history; remove duplicate records

Repeated events are legitimate. Repeated **engagement IDs** are not. Set the duplicate policy after inspecting the evidence. The rule is tied to the identifier—not to whether two rows look similar.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;"><p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Turn on the duplicate-ID removal policy.</p><p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> Why is an identical-looking row different from a duplicated engagement ID?</p><div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;"><strong>💡 Hint:</strong> Preserve legitimate repeated events; remove only later records with an engagement ID already seen.</div></div>'))


⬇️ 🔒 Checking cell below: Check the invalid-row policy and clean-artifact counts. ⬇️

In [ ]:
REMOVE_DUPLICATE_IDS = False  # TODO

working["event_date_clean"] = pd.to_datetime(
    working["event_date"], errors="coerce", format="mixed"
)
duplicate_id = (
    working["engagement_id"].notna()
    & working.duplicated(subset="engagement_id", keep="first")
)
print("Duplicate engagement IDs:", duplicate_id.sum())
deduped = working.loc[~duplicate_id].copy() if REMOVE_DUPLICATE_IDS else working.copy()

In [ ]:
#@title
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;"><p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Verify the duplicate policy and date parsing before continuing. <strong>⬇️</strong></p></div>'))


C1. Create the school-summary handoff This block aggregates the validated event-level artifact to one auditable record per school for Lab 2.

In [ ]:
check("Duplicate engagement IDs are removed", REMOVE_DUPLICATE_IDS and deduped["engagement_id"].dropna().is_unique,
      "Set REMOVE_DUPLICATE_IDS=True; legitimate repeated events have different IDs.")
check("Four dates cannot be parsed", working["event_date_clean"].isna().sum() == 4)

### Coding-assistant challenge

Ask your coding assistant:

> Explain each rejection flag in this cell in plain language. Identify one plausible bad row that these rules would miss and one reason a row should be kept in an exception queue rather than silently deleted. Do not change the code.

## B2. Make invalidity explicit

Convert numeric fields, create explicit rejection flags, and choose whether invalid rows enter the model-ready artifact. Do not hide uncertainty in a made-up quality score: each exception needs a reason that an owner could investigate.

In [ ]:
numeric_fields = [
    "recruiter_hours", "contacts", "appointments", "qualified", "contracts",
    "access_score", "distance_miles",
]
for field in numeric_fields:
    deduped[field] = pd.to_numeric(deduped[field], errors="coerce")

deduped["missing_key"] = (
    deduped["engagement_id"].isna()
    | deduped["school_id"].isna()
    | deduped["action_clean"].isna()
)
deduped["invalid_date"] = deduped["event_date_clean"].isna()
deduped["missing_numeric"] = deduped[numeric_fields].isna().any(axis=1)
deduped["negative_value"] = deduped[numeric_fields].lt(0).any(axis=1)
deduped["invalid_funnel"] = ~(
    deduped["contacts"].ge(deduped["appointments"])
    & deduped["appointments"].ge(deduped["qualified"])
    & deduped["qualified"].ge(deduped["contracts"])
)

flag_columns = ["missing_key", "invalid_date", "missing_numeric", "negative_value", "invalid_funnel"]
deduped["is_valid"] = ~deduped[flag_columns].any(axis=1)
validation_summary = deduped[flag_columns + ["is_valid"]].agg(["sum"]).T
validation_summary.columns = ["row_count"]
validation_summary

⬇️ 🔒 Checking cell below: Verify the summary before treating it as a Lab 2 handoff. ⬇️

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background: linear-gradient(135deg, rgba(245, 158, 11, 0.18), rgba(59, 130, 246, 0.10)); border: 1px solid rgba(245, 158, 11, 0.55); border-left: 6px solid #f59e0b; border-radius: 8px; padding: 14px 16px; margin: 10px 0 12px;"><p style="font-size: 1.05em; margin: 0 0 10px;"><strong>🛠️ TODO:</strong> Exclude invalid records from the model-ready artifact.</p><p style="margin: 0 0 10px;"><strong>🤔 Think:</strong> Why should invalid rows stay in the exception queue even when they are excluded from the clean dataset?</p><div style="background-color: rgba(59, 130, 246, 0.12); border-left: 4px solid #3b82f6; border-radius: 4px; padding: 9px 11px;"><strong>💡 Hint:</strong> Change the policy string to <code>exclude</code>; the rejection queue is created separately below.</div></div>'))


C2. Verify the prepared handoff These acceptance tests compare your artifacts with the prepared classroom versions; use them to verify the result and explain the policies behind it.

In [ ]:
INVALID_ROW_POLICY = "keep"  # TODO: change to "exclude"

selected = deduped.loc[deduped["is_valid"]].copy() if INVALID_ROW_POLICY == "exclude" else deduped.copy()
clean_columns = [
    "engagement_id", "event_date_clean", "school_id", "school_name_clean", "action_clean",
    "recruiter_hours", "contacts", "appointments", "qualified", "contracts",
    "access_score", "distance_miles",
]
clean_events = selected[clean_columns].rename(columns={
    "event_date_clean": "event_date",
    "school_name_clean": "school_name",
    "action_clean": "action",
})

integer_columns = ["recruiter_hours", "contacts", "appointments", "qualified", "contracts", "distance_miles"]
if INVALID_ROW_POLICY == "exclude":
    clean_events[integer_columns] = clean_events[integer_columns].astype(int)
clean_events = clean_events.sort_values(["event_date", "engagement_id"]).reset_index(drop=True)
clean_events.head()

In [ ]:
#@title
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;"><p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Check the invalid-row policy and clean-artifact counts. <strong>⬇️</strong></p></div>'))


⬇️ 🔒 Checking cell below: Check the invalid-row policy and clean-artifact counts. ⬇️

In [ ]:
check("Invalid records are excluded", INVALID_ROW_POLICY == "exclude")
check("Twenty-three unique records are rejected", (~deduped["is_valid"]).sum() == 23)
check("The clean event artifact contains 459 rows", len(clean_events) == 459)
check("Every clean row obeys the funnel",
      (clean_events["contacts"] >= clean_events["appointments"]).all()
      and (clean_events["appointments"] >= clean_events["qualified"]).all()
      and (clean_events["qualified"] >= clean_events["contracts"]).all())

In [ ]:
rejection_queue = deduped.loc[~deduped["is_valid"], [
    "engagement_id", "school_name", "action", "event_date", *flag_columns
]].copy()
print(f"Exception queue: {len(rejection_queue)} records requiring correction or review")
rejection_queue.head()

# Lab C — Publish and gate the data product

## C1. Create the school summary

Aggregate all valid events by school. Rates are intentionally left for Lab 2 to calculate. The summary must reconcile exactly to the approved event-level artifact.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(128, 128, 128, 0.12); border: 1px solid rgba(128, 128, 128, 0.28); border-radius: 6px; padding: 12px 16px; margin: 8px 0 12px;"><h3 style="margin: 0 0 6px;">C1. Create the school-summary handoff</h3><p style="margin: 0;">This block aggregates the validated event-level artifact to one auditable record per school for Lab 2.</p></div>'))


C1. Create the school-summary handoff This block aggregates the validated event-level artifact to one auditable record per school for Lab 2.

In [ ]:
school_summary = (
    clean_events
    .groupby(["school_id", "school_name"], as_index=False)
    .agg(
        historical_events=("engagement_id", "count"),
        recruiter_hours=("recruiter_hours", "sum"),
        contacts=("contacts", "sum"),
        appointments=("appointments", "sum"),
        qualified=("qualified", "sum"),
        contracts=("contracts", "sum"),
        access_score=("access_score", "first"),
        distance_miles=("distance_miles", "first"),
    )
    .sort_values("school_id")
    .reset_index(drop=True)
)
school_summary.head(8)

In [ ]:
#@title
display(HTML('<div style="background-color: rgba(16, 185, 129, 0.12); border: 1px solid rgba(16, 185, 129, 0.45); border-left: 6px solid #10b981; border-radius: 8px; padding: 11px 14px; margin: 8px 0 10px;"><p style="margin: 0; text-align: center;"><strong>⬇️ 🔒 Checking cell below:</strong> Verify the summary before treating it as a Lab 2 handoff. <strong>⬇️</strong></p></div>'))


⬇️ 🔒 Checking cell below: Verify the summary before treating it as a Lab 2 handoff. ⬇️

In [ ]:
check("The summary contains 36 schools", len(school_summary) == 36)
check("The summary has one row per school", school_summary["school_id"].is_unique)
check("Summary event counts reconcile to clean events", school_summary["historical_events"].sum() == len(clean_events))
check("No arbitrary data-quality score is present", "data_quality" not in school_summary.columns)

## C2. Verify the handoff

Lab 2 includes validated copies so it remains runnable even if a team does not finish Lab 1. Treat these comparisons as acceptance tests: they verify the artifact, but you should still be able to explain the policy behind every exclusion.

In [ ]:
#@title
from IPython.display import HTML, display
display(HTML('<div style="background-color: rgba(128, 128, 128, 0.12); border: 1px solid rgba(128, 128, 128, 0.28); border-radius: 6px; padding: 12px 16px; margin: 8px 0 12px;"><h3 style="margin: 0 0 6px;">C2. Verify the prepared handoff</h3><p style="margin: 0;">These acceptance tests compare your artifacts with the prepared classroom versions; use them to verify the result and explain the policies behind it.</p></div>'))


C2. Verify the prepared handoff These acceptance tests compare your artifacts with the prepared classroom versions; use them to verify the result and explain the policies behind it.

In [ ]:
expected_clean = pd.read_csv("https://usard-demo.netlify.app/data/clean_recruiting_events.csv", parse_dates=["event_date"])
expected_summary = pd.read_csv("https://usard-demo.netlify.app/data/school_summary.csv")

def frames_match(left, right):
    try:
        pd.testing.assert_frame_equal(left.reset_index(drop=True), right.reset_index(drop=True), check_dtype=False)
        return True
    except AssertionError:
        return False

check("Clean events match the prepared artifact", frames_match(clean_events, expected_clean))
check("School summary matches the prepared artifact", frames_match(school_summary, expected_summary))

## Mission debrief

- Lab 2A loads `school_summary.csv` to rank schools.
- Lab 2B loads `clean_recruiting_events.csv` to recommend actions.

Before Lab 2, issue a readiness decision: is this data product fit to support a school-ranking and action-recommendation exercise? Cite the identity rules, validation checks, reconciliation, and exception record.

**Aha:** Operationalizing data is the gate between a good requirement and a credible algorithm. AI can help write the transformation, but it cannot decide the authoritative definition, the correction policy, or whether the evidence is sufficient.

**Transition:** Lab 2 uses `school_summary.csv` to rank schools and `clean_recruiting_events.csv` to recommend actions.

**Next notebook:** [Lab 2: Recommender Systems](https://colab.research.google.com/github/TheDeafOne/USARD-AI/blob/main/labs/Lab_2_Recommender_Systems.ipynb)